# Seeding a code

When we start with the group $\langle ZZI, IZZ \rangle$, we can add the stabilizer $XXXX$ (along with a noiseless qubit) to correct phase-flip errors on the first three qubits. What if, instead, the original generators initially have support on the new qubits? We will try this out using $ZZIXI$ and $IZZIX$. Then we will add stabilizers to try to correct arbitrary single-qubit errors.

In [73]:
from typing import List
import numpy as np
import stim
import networkx as nx
from stimcirq import stim_circuit_to_cirq_circuit, cirq_circuit_to_stim_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim, stim_pauli_string_to_cirq
from encoded.diagonalizing_circuit import get_measurement_circuit

In [74]:
generators = [
    stim.PauliString("XXIIZZ"),
    stim.PauliString("IIIZIZ"),
    stim.PauliString("XZXXIX")
]

In [75]:
def all_single_qubit_errors(n: int) -> List[stim.PauliString]:
    errs = []
    for i in range(n):
        for p in range(1, 4):
            pauli_mask = [0] * i + [p] + [0] * (n - i - 1)
            errs.append(stim.PauliString(pauli_mask))
    return errs

In [76]:
errors = all_single_qubit_errors(6)
for err in errors:
    print(err)

+X_____
+Y_____
+Z_____
+_X____
+_Y____
+_Z____
+__X___
+__Y___
+__Z___
+___X__
+___Y__
+___Z__
+____X_
+____Y_
+____Z_
+_____X
+_____Y
+_____Z


In [77]:
def get_uncorrectable_errors(generators):
    number_true = 0
    number_checked = 0
    uncorrectable_errors = []
    for i, ei in enumerate(errors):
        for j in range(i):
            number_checked += 1
            ej = errors[j]
            e = ei * ej
            commutators = []
            for generator in generators:
                comm = e.commutes(generator)
                commutators.append(comm)
            has_anticommuting_operator = any([not b for b in commutators])
            if has_anticommuting_operator:
                number_true += 1
            else:
                print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
                uncorrectable_errors.append(e)
    print(f"{number_true}/{number_checked} operators anticommute.")
    return uncorrectable_errors

In [78]:
errors_good = get_uncorrectable_errors(generators)

+Z_____ * +Y_____ = -iX_____, [True, True, True] False 
+_Y____ * +Y_____ = +YY____, [True, True, True] False 
+_Y____ * +Z_____ = +ZY____, [True, True, True] False 
+__X___ * +X_____ = +X_X___, [True, True, True] False 
+__Y___ * +_X____ = +_XY___, [True, True, True] False 
+__Z___ * +_X____ = +_XZ___, [True, True, True] False 
+__Z___ * +__Y___ = -i__X___, [True, True, True] False 
+___Z__ * +_X____ = +_X_Z__, [True, True, True] False 
+___Z__ * +__Y___ = +__YZ__, [True, True, True] False 
+___Z__ * +__Z___ = +__ZZ__, [True, True, True] False 
+____X_ * +_Z____ = +_Z__X_, [True, True, True] False 
+____Y_ * +_Z____ = +_Z__Y_, [True, True, True] False 
+____Y_ * +____X_ = -i____Z_, [True, True, True] False 
+____Z_ * +X_____ = +X___Z_, [True, True, True] False 
+____Z_ * +__X___ = +__X_Z_, [True, True, True] False 
+_____Z * +_X____ = +_X___Z, [True, True, True] False 
+_____Z * +__Y___ = +__Y__Z, [True, True, True] False 
+_____Z * +__Z___ = +__Z__Z, [True, True, True] False 
+_____Z

In [79]:
generators = [
    stim.PauliString("XXIIZZ"),
    stim.PauliString("IIIZIZ"),
    stim.PauliString("IZIIIX")
]

In [80]:
errors_bad = get_uncorrectable_errors(generators)

+Z_____ * +Y_____ = -iX_____, [True, True, True] False 
+_Z____ * +Y_____ = +YZ____, [True, True, True] False 
+_Z____ * +Z_____ = +ZZ____, [True, True, True] False 
+__X___ * +X_____ = +X_X___, [True, True, True] False 
+__Y___ * +X_____ = +X_Y___, [True, True, True] False 
+__Y___ * +__X___ = -i__Z___, [True, True, True] False 
+__Z___ * +X_____ = +X_Z___, [True, True, True] False 
+__Z___ * +__X___ = +i__Y___, [True, True, True] False 
+__Z___ * +__Y___ = -i__X___, [True, True, True] False 
+___Y__ * +___X__ = -i___Z__, [True, True, True] False 
+___Z__ * +X_____ = +X__Z__, [True, True, True] False 
+___Z__ * +__X___ = +__XZ__, [True, True, True] False 
+___Z__ * +__Y___ = +__YZ__, [True, True, True] False 
+___Z__ * +__Z___ = +__ZZ__, [True, True, True] False 
+____X_ * +Y_____ = +Y___X_, [True, True, True] False 
+____X_ * +Z_____ = +Z___X_, [True, True, True] False 
+____X_ * +_Z____ = +_Z__X_, [True, True, True] False 
+____Y_ * +Y_____ = +Y___Y_, [True, True, True] False 
+____

In [81]:
bad_strs = [str(err) for err in errors_bad]
good_strs = [str(err) for err in errors_good]
assert set(bad_strs).issubset(good_strs)
diff = set(good_strs) - set(bad_strs)
for err in list(diff):
    print(err)

AssertionError: 

In [82]:
print(len(errors_good))
print(len(errors_bad))

19
27
